In [ ]:
'''
Done: getting ndcg-value for debateorg.json 
Running: getting debateorg corpus re-created by deepct
'''

In [ ]:
!pip install python-terrier
!pip install json
import pyterrier as pt
if not pt.started():
  pt.init()

In [ ]:
from typing import List, Set, Callable

def tokenize_text(text: str, stopwords: Set[str] = None) -> List[str]:
    if stopwords is None:
        stopwords = set()

    raw_tokens = re.split('\W', text)
    normalized_tokens = map(lambda w: w.lower().strip(".:,;?!"), raw_tokens)
    return ' '.join(list(filter(lambda w: w != '' and w not in stopwords, normalized_tokens)))

In [ ]:
import ijson
import re
data_set = set()
id_text = {}
# only parliamentary.json from args.me corpus
# filename = "/content/drive/MyDrive/workspace/parliamentary.json"
filename = "/content/drive/MyDrive/workspace/debateorg.json"
with open(filename, 'r', encoding='utf-8') as f:
    objects = ijson.items(f, 'arguments.item')
    for i, o in enumerate(objects):
        id_text[i] = str(i), o['id'], tokenize_text(o["premises"][0]["text"])

In [ ]:
import pandas as pd
test_df = pd.DataFrame.from_dict(id_text).transpose()
test_df = test_df.rename(columns={0: "docid", 1: "docno", 2: "text"})

In [ ]:
pd_indexer = pt.DFIndexer("./pd_index")
indexref = pd_indexer.index(test_df["text"], test_df["docno"])

In [ ]:
from xml.etree import ElementTree
tree = ElementTree.parse('/content/drive/MyDrive/workspace/topics.xml')
data = tree.getroot()
titles = []
title_ids = []
for topic in data:
    titles.append(topic.find('title').text.lower()[:-1])
    num = topic.find('number').text
    title_ids.append(num)


In [ ]:
topics = pd.DataFrame(titles,columns = ['query'])
topics['qid'] = topics.index + 51
columns_titles = ["qid","query"]
topics=topics.reindex(columns=columns_titles)

In [ ]:
index = pt.IndexFactory.of(indexref)
retr = pt.BatchRetrieve(index, controls = {"wmodel": "TF_IDF"})
res=retr.transform(topics)

In [ ]:
qrels_path=("/content/drive/MyDrive/workspace/qrels.qrels")
qrels = pt.io.read_qrels(qrels_path)

In [ ]:
eval = pt.Utils.evaluate(res,qrels,metrics=["map", "ndcg"])
eval

In [ ]:
import os
if not os.path.exists("uncased_L-12_H-768_A-12.zip"):
  !wget https://storage.googleapis.com/bert_models/2020_02_20/uncased_L-12_H-768_A-12.zip
  !unzip uncased_L-12_H-768_A-12.zip
  !mkdir -p bert-base-uncased
  !mv vocab.txt bert_* bert-base-uncased/

In [ ]:
%tensorflow_version 1.x

!pip install --upgrade git+https://github.com/terrier-org/pyterrier.git#egg=python-terrier
!pip install --upgrade git+https://github.com/terrierteam/pyterrier_deepct.git

In [ ]:
if not os.path.exists("marco.zip"):
  !wget http://boston.lti.cs.cmu.edu/appendices/arXiv2019-DeepCT-Zhuyun-Dai/outputs/marco.zip
  !unzip marco.zip

In [ ]:
import pyterrier_deepct
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 
import tensorflow as tf
assert tf.__version__.startswith("1"), "TF 1 is required by DeepCT; on Colab, use %tensorflow_version 1.x"
tf.logging.set_verbosity(tf.logging.ERROR)
deepct = pyterrier_deepct.DeepCTTransformer("bert-base-uncased/bert_config.json", "marco/model.ckpt-65816")

In [ ]:
super_df = test_df.copy()
# super_df

In [ ]:
# !rm -rf /content/pd_super_index

In [ ]:
xd = test_df.head().copy()
xd

In [ ]:
deepct.transform(xd)

In [ ]:
super_df = deepct.transform(super_df)

In [ ]:
super_df.to_csv(index=False, path_or_buf='/content/drive/MyDrive/workspace/superdf.csv')

In [ ]:
super_df

In [ ]:
pd_super_indexer = pt.DFIndexer("./pd_super_index")
index_super_ref = pd_super_indexer.index(super_df["text"], super_df["docno"])

In [ ]:
index_super = pt.IndexFactory.of(index_super_ref)

In [ ]:
retr_super = pt.BatchRetrieve(index_super, controls = {"wmodel": "BM25"})
res_super=retr_super.transform(topics)

In [ ]:
res_super

In [ ]:
# # dataset = pt.get_dataset("irds:vaswani")
# # index_loc = "./deepct_index_path"
# indexer = deepct >> pd_indexer
# indexref = pd_indexer.index(test_df["text"], test_df["docno"])
